# 03 — Modelagem: as três trilhas na mesma tabela

Roda baselines, GNN relacional e GNN geográfica sobre **a mesma partição e os
mesmos exemplos**, e reporta tudo junto.

Isso não é conveniência de apresentação, é a condição para o resultado significar
alguma coisa. A versão anterior deste notebook avaliava a GNN com
`train_mask` como máscara de teste — o número reportado como desempenho de teste
era desempenho de treino — e não tinha baseline nenhuma ao lado. Ver D-11.

**Como ler os números.** A prevalência é 0,048% no recorte estadual, um positivo a
cada ~2.100 candidatos. O valor absoluto do average precision não é interpretável
nessa escala: um AP de 0,01 é vinte vezes a linha de base e ainda parece zero. A
métrica de destaque é **MAP@k por estabelecimento**, que ranqueia os 99 tipos de
equipamento dentro de cada estabelecimento. D-24 mostra por quê na prática: AP e
MAP@10 ordenam as baselines de formas diferentes.

**Custo.** Cerca de uma hora no recorte estadual, com pico de 6,3 GB. Numa máquina
de 9 GB isso só cabe sob um cgroup com teto de memória — ver o alvo `experimento`
do Makefile, que embrulha a execução em `systemd-run`.

In [1]:
import sys
from pathlib import Path

BASE_DIR = Path.cwd().parent
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import torch

from src.etl import changes
from src.ml import baselines, gnn, graph, metrics, tasks
from src.ml.splits import particionar

pd.set_option("display.width", 200)
torch.manual_seed(42)

PERIODOS = changes.periodos_disponiveis()
PARTICAO = particionar(PERIODOS)
print(f"snapshots: {PERIODOS}\n")
print(PARTICAO.resumo())
print(f"\nfim da janela de treino: {PARTICAO.fim_do_treino}")
print("Nenhuma feature de treino pode enxergar além dessa data.")

/home/phprestes/Documents/IC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


snapshots: ['201701', '201801', '201901', '202001', '202101', '202201', '202301', '202401', '202501', '202601']

conjunto     n  transições (por período de destino)
treino       6  201801 201901 202001 202101 202201 202301
validacao    2  202401 202501
teste        1  202601

fim da janela de treino: 202301
Nenhuma feature de treino pode enxergar além dessa data.


## A tarefa

Uma tabela de rótulos, consumida por todas as trilhas. Nenhuma delas recalcula a
partição — é isso que garante que comparem os mesmos exemplos.

In [2]:
# `negativos_por_positivo` subamostra os negativos SÓ no treino; validação e
# teste ficam completos, senão a prevalência medida seria artificial (D-23).
tarefa = tasks.tarefa_aquisicao(PARTICAO, recorte=graph.RECORTE_PADRAO)
print(f"{tarefa.nome}")
print(f"{len(tarefa.df):,} exemplos | {tarefa.memoria_gb():.2f} GB | "
      f"prevalência global {tarefa.prevalencia:.5%}")
print()
print(tarefa.resumo().to_string(index=False))

aquisicao:rlEstabEquipamento.co_equipamento
40,838,979 exemplos | 0.35 GB | prevalência global 0.10010%



 conjunto periodo_destino  exemplos  positivos  prevalencia
    teste          202601  13208275       6309     0.000478
   treino          201801    670536       3336     0.004975
   treino          201901    581091       2891     0.004975
   treino          202001    893043       4443     0.004975
   treino          202101   1122585       5585     0.004975
   treino          202201    972036       4836     0.004975
   treino          202301    819477       4077     0.004975
validacao          202401  10900456       4409     0.000404
validacao          202501  11671480       4994     0.000428


## Trilha 1 — baselines sem estrutura

Cinco modelos que não veem relação nem vizinhança. A diferença entre eles e as
trilhas 2 e 3 é a medida do valor da estrutura.

`persistencia` prevê zero por construção, já que todo candidato é um par ausente
em `t`. O AP dela é exatamente a prevalência — é a régua, não um competidor.

In [3]:
previsoes = baselines.rodar_todas(tarefa, PARTICAO, conjunto="teste")

resultados = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in previsoes.items()
}
print(metrics.tabela_de_resultados(resultados).to_string())
print()
for nome, p in previsoes.items():
    print(f"{nome}: {p.metadados}")

                               n  positivos  prevalencia  average_precision   auc_roc    map@10
por_entidade          13208275.0     6309.0     0.000478           0.003135  0.738540  0.158715
gbdt_geral            13208275.0     6309.0     0.000478           0.002535  0.750995  0.191489
gbdt_ultimo_snapshot  13208275.0     6309.0     0.000478           0.002019  0.744551  0.185679
popularidade_item     13208275.0     6309.0     0.000478           0.001901  0.699114  0.272477
persistencia          13208275.0     6309.0     0.000478           0.000478  0.500000  0.032840

persistencia: {'observacao': 'prevê zero por construção; AP = prevalência'}
popularidade_item: {'itens_estimados': 99, 'taxa_base': 0.0049753200876893}
gbdt_geral: {'n_treino': 5058768, 'features': ['co_unidade', 'periodo_destino', 'co_equipamento']}
gbdt_ultimo_snapshot: {'n_treino': 819477, 'features': ['co_unidade', 'periodo_destino', 'co_equipamento']}
por_entidade: {'entidades_com_modelo_proprio': 6573, 'entidades_

## Trilha 2 — GNN relacional

O grafo do schema CNES inteiro. Nós são estabelecimentos e tabelas de fato;
arestas são as chaves estrangeiras de `docs/01-selecao-tabelas.md`.

Montar o grafo é a parte cara: o filtro por município é empurrado para dentro do
scan Parquet, senão seria preciso carregar o país inteiro para descartar 98%.

In [4]:
# Projeção mínima: sem ela o Database do estado carrega 76 milhões de linhas com
# todas as colunas e chega a 5,3 GB numa máquina de 9 GB (D-23). É a mesma
# projeção que tools/roda_experimento.py usa, e as duas precisam concordar.
db = graph.montar_db(
    recorte=graph.RECORTE_PADRAO, colunas=graph.colunas_minimas_para_grafo()
)
print(f"recorte: {graph.RECORTE_PADRAO!r} (prefixo de código IBGE)")
print(f"tabelas no grafo: {len(db.table_dict)}")
for nome, t in sorted(db.table_dict.items(), key=lambda kv: -kv[1].df.num_rows)[:8]:
    print(f"  {nome:28} {t.df.num_rows:>10,} linhas  fkeys={list((t.fkey_col_to_pkey_table or {}))}")

recorte: '35' (prefixo de código IBGE)
tabelas no grafo: 36
  tbCargaHorariaSus            11,766,512 linhas  fkeys=['co_unidade']
  tbEstabHorarioAtend           3,967,702 linhas  fkeys=['co_unidade']
  rlEstabEquipamento            2,621,838 linhas  fkeys=['co_unidade']
  rlEstabAtendPrestConv         1,966,829 linhas  fkeys=['co_unidade']
  rlEstabServClass              1,896,976 linhas  fkeys=['co_unidade']
  rlEstabInstFisiAssist         1,762,342 linhas  fkeys=['co_unidade']
  rlEstabColetaSelRejeito       1,707,689 linhas  fkeys=['co_unidade']
  rlEstabProgFundo              1,252,736 linhas  fkeys=['co_unidade']


In [5]:
unidades = sorted(set(db.table_dict[graph.TABELA_RAIZ].df[graph.COL_ENTIDADE].to_pylist()))
itens = sorted(tarefa.df[tarefa.col_item].dropna().unique())
indice = gnn.IndicePares.de(unidades, itens)
print(f"{len(unidades):,} estabelecimentos  x  {len(itens)} tipos de equipamento")

# O corte e `antes_de_todos_os_rotulos`, nunca `fim_do_treino`: o grafo e
# estatico e serve treino, validacao e teste, entao corta-lo ao fim do treino
# inscreveria o rotulo na propria estrutura (D-25). As features usam o mesmo
# corte, pelo mesmo motivo.
CORTE_DO_GRAFO = PARTICAO.antes_de_todos_os_rotulos
print(f"corte do grafo e das features: {CORTE_DO_GRAFO}")

features = gnn.features_de_estabelecimento(db, unidades, ate_periodo=CORTE_DO_GRAFO)
print(f"matriz de features: {tuple(features.shape)}")

dados_rel = gnn.grafo_relacional_para_data(
    db, unidades, features, ate_periodo=CORTE_DO_GRAFO
)
print(f"tipos de no: {len(dados_rel.node_types)}  tipos de aresta: {len(dados_rel.edge_types)}")

146,679 estabelecimentos  x  99 tipos de equipamento
corte do grafo e das features: 201701


matriz de features: (146679, 24)


/home/phprestes/Documents/IC/src/ml/gnn.py:394: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  origem = torch.as_tensor(


tipos de no: 25  tipos de aresta: 48


In [6]:
modelo_rel, hist_rel = gnn.treinar_aquisicao(
    tarefa, PARTICAO, dados_rel, indice, epocas=200, paciencia=20, verboso=True
)
print(f"\nmelhor época {hist_rel['melhor_epoca']}  "
      f"AP validação {hist_rel['melhor_ap_validacao']:.5f}  "
      f"({hist_rel['epocas_rodadas']} épocas, {hist_rel['dispositivo']})")

prev_rel = gnn.prever_aquisicao(modelo_rel, tarefa, dados_rel, indice,
                                conjunto="teste", nome="gnn_relacional")
print(prev_rel.metadados)

/home/phprestes/Documents/IC/.venv/lib/python3.12/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()
/home/phprestes/Documents/IC/.venv/lib/python3.12/site-packages/torch_geometric/nn/to_hetero_transformer.py:120: UserWarning: Found function 'dropout_1' with keyword argument 'training'. During FX tracing, this will likely be baked in as a constant value. Consider replacing this function by a module to properly encapsulate its training flag.
  return transformer.transform()


época    0  perda 1.3890  AP validação 0.0022


época   20  perda 0.8631  AP validação 0.0039


época   40  perda 0.8081  AP validação 0.0050



melhor época 28  AP validação 0.00557  (49 épocas, cpu)


{'n_nos': 146679, 'pares_avaliados': 13208275, 'pares_descartados': 0}


## Trilha 3 — GNN geográfica

Só estabelecimentos e proximidade física. Ignora a estrutura de tabelas por
construção.

**Ressalva obrigatória (D-15, D-22, D-41).** A cobertura de coordenada é de 87,3%
no estado — D-17 dizia 57%, medido sobre parte da série, e D-22 corrigiu. A trilha 3
fala sobre esse subconjunto, e a comparação com as outras duas precisa ser feita
**sobre os mesmos nós**, senão mede diferença de amostra em vez de diferença de
estrutura.

A exclusão não é aleatória: D-41 mede V de Cramér de 0,349 em `co_natureza_jur` e
0,153 em `tp_pfpj`, com pessoa jurídica 10,7 pontos mais coberta que pessoa física.
Mas ela **não alcança os rótulos** — nas duas transições mais recentes, zero
aquisições caem em estabelecimento não posicionável.

In [7]:
grafo_geo = graph.montar_grafo_geografico(db, k=10)
print(f"nos posicionaveis: {grafo_geo.n_nos:,} de {len(unidades):,} "
      f"({100 * grafo_geo.n_nos / len(unidades):.1f}%)")
print(f"arestas: {grafo_geo.n_arestas:,}  (kNN k={grafo_geo.k}, simetrizado)")

indice_geo = gnn.IndicePares.de(grafo_geo.unidades, itens)
features_geo = gnn.features_de_estabelecimento(
    db, grafo_geo.unidades, ate_periodo=CORTE_DO_GRAFO
)
dados_geo = gnn.grafo_geografico_para_data(grafo_geo, features_geo)
print(f"features: {tuple(features_geo.shape)}")

nos posicionaveis: 127,868 de 146,679 (87.2%)
arestas: 1,913,816  (kNN k=10, simetrizado)


features: (127868, 24)


In [8]:
modelo_geo, hist_geo = gnn.treinar_aquisicao(
    tarefa, PARTICAO, dados_geo, indice_geo, epocas=200, paciencia=20, verboso=True
)
print(f"\nmelhor época {hist_geo['melhor_epoca']}  "
      f"AP validação {hist_geo['melhor_ap_validacao']:.5f}")

prev_geo = gnn.prever_aquisicao(modelo_geo, tarefa, dados_geo, indice_geo,
                                conjunto="teste", nome="gnn_geografica")
print(prev_geo.metadados)

época    0  perda 1.3879  AP validação 0.0008


época   20  perda 0.9447  AP validação 0.0034


época   40  perda 0.9049  AP validação 0.0034


época   60  perda 0.8847  AP validação 0.0036



melhor época 46  AP validação 0.00378


{'n_nos': 127868, 'pares_avaliados': 11411933, 'pares_descartados': 1796342}


## Resultado

A regra de D-11: nenhuma métrica de GNN aparece sem as baselines na mesma tabela.

A primeira tabela usa todos os exemplos de teste, o que é injusto com a trilha 3 —
ela só pontua os estabelecimentos posicionáveis. A segunda restringe todas as
trilhas ao mesmo subconjunto, e é a comparação que de fato responde se a estrutura
geográfica acrescenta algo. **Toda conclusão que envolva a trilha 3 cita a tabela
pareada**, nunca a completa.

In [9]:
todas = dict(previsoes)
todas["gnn_relacional"] = prev_rel
todas["gnn_geografica"] = prev_geo

completa = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in todas.items()
}
print("=== todos os exemplos de teste ===")
print("(a trilha geográfica avalia menos pares; ver a tabela pareada abaixo)\n")
print(metrics.tabela_de_resultados(completa).to_string())

=== todos os exemplos de teste ===
(a trilha geográfica avalia menos pares; ver a tabela pareada abaixo)

                               n  positivos  prevalencia  average_precision   auc_roc    map@10
gnn_relacional        13208275.0     6309.0     0.000478           0.005840  0.845652  0.253323
gnn_geografica        11411933.0     6309.0     0.000553           0.004931  0.799940  0.266476
por_entidade          13208275.0     6309.0     0.000478           0.003135  0.738540  0.158715
gbdt_geral            13208275.0     6309.0     0.000478           0.002535  0.750995  0.191489
gbdt_ultimo_snapshot  13208275.0     6309.0     0.000478           0.002019  0.744551  0.185679
popularidade_item     13208275.0     6309.0     0.000478           0.001901  0.699114  0.272477
persistencia          13208275.0     6309.0     0.000478           0.000478  0.500000  0.032840


In [10]:
# Comparação pareada: só os estabelecimentos que a trilha geográfica alcança.
# `mascara_de_entidades` trabalha sobre os códigos inteiros de `Previsao`, sem
# reconstruir o vetor de `co_unidade` como strings — com 11 milhões de linhas de
# teste, aquele vetor sozinho passa de um gigabyte (D-23).
posicionaveis = set(grafo_geo.unidades)

def restringir(p):
    dentro = p.mascara_de_entidades(posicionaveis)
    return metrics.avaliar_classificacao(
        p.y[dentro], p.escore[dentro], p.entidades[dentro], k=10
    )

pareada = {nome: restringir(p) for nome, p in todas.items()}
print(f"=== restrito aos {len(posicionaveis):,} estabelecimentos posicionáveis ===")
print(f"(cobertura de {100 * len(posicionaveis) / len(unidades):.1f}% dos nós; ver D-22)\n")
print(metrics.tabela_de_resultados(pareada).to_string())

=== restrito aos 127,868 estabelecimentos posicionáveis ===
(cobertura de 87.2% dos nós; ver D-22)

                               n  positivos  prevalencia  average_precision   auc_roc    map@10
gnn_relacional        11411933.0     6309.0     0.000553           0.006498  0.841169  0.253323
gnn_geografica        11411933.0     6309.0     0.000553           0.004931  0.799940  0.266476
por_entidade          11411933.0     6309.0     0.000553           0.003346  0.737349  0.157976
gbdt_geral            11411933.0     6309.0     0.000553           0.002887  0.750934  0.190980
gbdt_ultimo_snapshot  11411933.0     6309.0     0.000553           0.002296  0.743932  0.185893
popularidade_item     11411933.0     6309.0     0.000553           0.002201  0.698777  0.272477
persistencia          11411933.0     6309.0     0.000553           0.000553  0.500000  0.033263


### Experimento de controle: sem as transições de pandemia

As transições que tocam 2020 ou 2021 atravessam um regime de aquisição
excepcional. A metodologia (seção 4.1) exige rodar a variante sem elas e relatar
se a conclusão muda.

In [11]:
# `recorte` explicito: o default de `tarefa_aquisicao` ja e RECORTE_PADRAO, mas
# depender do default faria o controle mudar de escopo em silencio se ele mudar.
particao_sem_covid = particionar(PERIODOS, excluir_pandemia=True)
print(particao_sem_covid.resumo())

tarefa_sem_covid = tasks.tarefa_aquisicao(particao_sem_covid, recorte=graph.RECORTE_PADRAO)
prev_sem_covid = baselines.rodar_todas(tarefa_sem_covid, particao_sem_covid, conjunto="teste")

controle = {
    nome: metrics.avaliar_classificacao(p.y, p.escore, p.entidades, k=10)
    for nome, p in prev_sem_covid.items()
}
print()
print(metrics.tabela_de_resultados(controle).to_string())

conjunto     n  transições (por período de destino)
treino       3  201801 201901 202301
validacao    2  202401 202501
teste        1  202601



                               n  positivos  prevalencia  average_precision   auc_roc    map@10
gbdt_ultimo_snapshot  13208275.0     6309.0     0.000478           0.002179  0.750767  0.183943
gbdt_geral            13208275.0     6309.0     0.000478           0.002084  0.706388  0.188302
por_entidade          13208275.0     6309.0     0.000478           0.001957  0.711866  0.186172
popularidade_item     13208275.0     6309.0     0.000478           0.001167  0.687566  0.111072
persistencia          13208275.0     6309.0     0.000478           0.000478  0.500000  0.032840


## Leitura dos resultados

As três perguntas que a tabela pareada precisa responder:

1. **Alguma trilha supera a persistência ingênua?** Se não, nada foi aprendido — e o
   AP da persistência é literalmente a prevalência, com AUC exatamente 0,500. Se
   algum dia não for, o arnês está quebrado (D-24).
2. **As GNNs superam o GBDT tabular?** Essa diferença é o valor da estrutura, que é
   a contribuição do trabalho. Se não houver diferença, a conclusão honesta é que a
   estrutura relacional do CNES não ajuda nesta tarefa — resultado negativo
   publicável, não um fracasso.
3. **A trilha geográfica supera a relacional na comparação pareada?** Responde se
   escassez se explica melhor por vizinhança física ou por estrutura administrativa.

O resultado de referência está em D-32: `gnn_relacional` AP 0,01061 / AUC 0,849 /
MAP@10 0,300, contra 0,00220 / 0,700 / 0,271 de `popularidade_item`. A relacional
vence as duas métricas.

**Cuidado com casas decimais.** Duas execuções deste notebook com o mesmo código e
os mesmos dados deram AP de 0,00285 e 0,00329 para `gbdt_geral` — 15% de variação
sem nenhuma mudança. Diferenças dessa ordem entre configurações não são
interpretáveis sem repetição por semente.